<a href="https://colab.research.google.com/github/FrancoPanico/glioblastoma-mri-seg-mgmt/blob/Z-score/z_score_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#importing libraries for data normalization

import nibabel as nib
import numpy as np
import pandas as pd
import shutil
import os
import matplotlib.pyplot as plt

In [9]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#getting the datas from the drive and understand how they are organized

data_root = Path("/content/drive/MyDrive/Assignment_26_27/training_images")
out_root = Path("/content/normalized_scans")
out_root.mkdir(parents=True, exist_ok=True)

SCAN_FILENAMES = [
    "t1.nii.gz",
    "t1ce.nii.gz",
    "t2.nii.gz",
    "flair.nii.gz",
 ]

SEG_FILENAME = "seg.nii.gz"

def zscore_foreground(volume: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    vol = volume.astype(np.float32)
    mask = vol > 0
    if not np.any(mask):
        return np.zeros_like(vol, dtype=np.float32)

    fg = vol[mask]
    mean = fg.mean()
    std = fg.std()

    out = np.zeros_like(vol, dtype=np.float32)
    out[mask] = (fg - mean) / max(std, eps)
    return out

patient_dirs = sorted([p for p in data_root.iterdir() if p.is_dir()])
print(f"Found {len(patient_dirs)} patient folders in total.")
print("Processing all patients...")

for idx, p_dir in enumerate(patient_dirs):
    p_out = out_root / p_dir.name
    p_out.mkdir(parents=True, exist_ok=True)

    # 1. Normalize the 4 MRI scans
    for fname in SCAN_FILENAMES:
        in_path = p_dir / fname
        if not in_path.exists():
            print(f"[Missing scan] {p_dir.name}/{fname}")
            continue

        img = nib.load(in_path)
        norm_data = zscore_foreground(img.get_fdata(dtype=np.float32))
        norm_img = nib.Nifti1Image(norm_data, affine=img.affine, header=img.header)
        nib.save(norm_img, p_out / fname)

    # 2. Copy seg.nii.gz without altering label values
    seg_in = p_dir / SEG_FILENAME
    if seg_in.exists():
        shutil.copy2(seg_in, p_out / SEG_FILENAME)
    else:
        print(f"[Missing seg] {p_dir.name}/{SEG_FILENAME}")

    if (idx + 1) % 50 == 0 or (idx + 1) == len(patient_dirs):
        print(f"Progress: {idx + 1}/{len(patient_dirs)} patients processed.")

print("Processing complete for the entire dataset!")

Found 1384 patient folders in total.
Processing all patients...
Progress: 50/1384 patients processed.
Progress: 100/1384 patients processed.
